# E02 Readiness and Correctness Notebook

This notebook is for **E02 – Task pooling experiment**.

## Scientific question

Should the thesis use the `coarse_affect` target:
- **pooled across tasks**, or
- **separately by task**?

## Fixed controls

This notebook assumes the current E02 primary setup is:

- target: `coarse_affect`
- model: `ridge`
- cv: `within_subject_loso_session`

## Main purpose

This notebook is designed to answer two questions before running or interpreting E02:

1. **Readiness**  
   Is E02 operationally ready to run through the project stack?

2. **Correctness**  
   Is E02 scientifically well-specified, and will the comparison mean what we think it means?

## Expected answer pattern

By the end of this notebook you should know:

- whether the **registry path** is ready
- whether the **workbook path** is ready
- whether the dataset supports the **task-specific** comparison fairly
- whether all tasks should be included, or whether a narrower task whitelist is needed
- what commands to use for dry-run, smoke-run, and full E02 execution


## Important interpretation note

E02 is **not** a confirmatory result. It is a **method-choice experiment** for Stage 1 target lock / pooling policy.

A task-specific comparison is only scientifically useful if:

- tasks have enough samples
- tasks have enough class coverage
- held-out-session LOSO remains valid within each task and subject
- the tasks being compared are conceptually the intended task families, not arbitrary task labels from the dataset index


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
from pprint import pprint

import pandas as pd

REPO_ROOT = Path("..").resolve()
REGISTRY = REPO_ROOT / "configs/decision_support_registry.json"
WORKBOOK = REPO_ROOT / "templates/thesis_experiment_program_revised.xlsx"
INDEX_CSV = REPO_ROOT / "Data/processed/dataset_index.csv"
DATA_ROOT = REPO_ROOT / "Data"
CACHE_DIR = REPO_ROOT / "Data/processed/feature_cache"
OUTPUT_ROOT = REPO_ROOT / "outputs/artifacts/decision_support"
EXPERIMENT_ID = "E02"

print("Repo root:", REPO_ROOT)
print("Registry exists:", REGISTRY.exists(), REGISTRY)
print("Workbook exists:", WORKBOOK.exists(), WORKBOOK)
print("Index exists:", INDEX_CSV.exists(), INDEX_CSV)
print("Data root exists:", DATA_ROOT.exists(), DATA_ROOT)
print("Cache dir exists:", CACHE_DIR.exists(), CACHE_DIR)


Repo root: D:\Khaled\Projects\VScode\Thesis
Registry exists: True D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json
Workbook exists: True D:\Khaled\Projects\VScode\Thesis\templates\thesis_experiment_program_revised.xlsx
Index exists: True D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv
Data root exists: True D:\Khaled\Projects\VScode\Thesis\Data
Cache dir exists: True D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache


## 1. Inspect the registry definition for E02

The registry is currently the most likely executable source of truth.


In [2]:
registry_data = json.loads(REGISTRY.read_text(encoding="utf-8"))
e02_entry = None
for item in registry_data.get("experiments", []):
    if item.get("experiment_id") == EXPERIMENT_ID:
        e02_entry = item
        break

pprint(e02_entry)


{'blocked_reasons': [],
 'dataset_scope': 'Internal BAS2 repeated-session index.',
 'decision_id': 'D01',
 'executable_now': True,
 'execution_status': 'executable',
 'experiment_id': 'E02',
 'fixed_controls': 'coarse_affect target, ridge model, '
                   'within_subject_loso_session split, seed=42.',
 'manipulated_factor': 'Pooling strategy by task',
 'models': ['ridge'],
 'notes': 'Compares pooled tasks vs per-task filtering.',
 'output_dir': 'outputs/artifacts/decision_support/E02',
 'primary_metric': 'balanced_accuracy',
 'secondary_metrics': ['macro_f1', 'accuracy'],
 'split_logic': 'within_subject_loso_session, per subject.',
 'stage': 'Stage 1 - Target lock',
 'target_definition': 'coarse_affect',
 'title': 'Task pooling experiment',
 'variant_templates': [{'expand': {'subject': 'subjects'},
                        'params': {'cv': 'within_subject_loso_session',
                                   'model': 'ridge',
                                   'target': 'coarse_a

### Readiness interpretation

What you want to confirm here:

- `executable_now = True`
- `execution_status = executable`
- pooled-task template exists
- task-specific template exists
- the fixed controls match the intended E02 design


## 2. Inspect the workbook definition for E02

This checks whether the workbook itself is executable for E02, or whether E02 only really works through the registry.


In [3]:
xl = pd.ExcelFile(WORKBOOK)

exp_defs = pd.read_excel(WORKBOOK, sheet_name="Experiment_Definitions")
master = pd.read_excel(WORKBOOK, sheet_name="Master_Experiments")
decisions = pd.read_excel(WORKBOOK, sheet_name="Decision_Log")
grouping = pd.read_excel(WORKBOOK, sheet_name="Grouping_Strategy_Map")
data_selection = pd.read_excel(WORKBOOK, sheet_name="Data_Selection_Design")

print("Experiment_Definitions row for E02:")
display(exp_defs[exp_defs["experiment_id"].astype(str).str.upper() == "E02"])

print("\nMaster_Experiments row for E02:")
display(master[master["Experiment_ID"].astype(str).str.upper() == "E02"])

print("\nDecision_Log rows mentioning E02:")
mask = decisions.astype(str).apply(lambda s: s.str.contains("E02", na=False))
display(decisions[mask.any(axis=1)])

print("\nGrouping_Strategy rows mentioning E02:")
mask = grouping.astype(str).apply(lambda s: s.str.contains("E02", na=False))
display(grouping[mask.any(axis=1)])

print("\nData_Selection rows relevant to pooled vs task-specific framing:")
display(data_selection[data_selection["Data_Slice_ID"].isin(["DS01", "DS03"])])


Experiment_Definitions row for E02:


,experiment_id,enabled,start_section,end_section,base_artifact_id,target,cv,model,subject,train_subject,test_subject,filter_task,filter_modality,reuse_policy,search_space_id
1,E02,Yes,dataset_selection,evaluation,NaN,coarse_affect,within_subject_loso_session,ridge,NaN,NaN,NaN,VARIES,NaN,require_explicit_base,NaN



Master_Experiments row for E02:


,Experiment_ID,Short_Title,Category,Evidential_Role,Stage,Priority,Decision_Supported,Exact_Question,Hypothesis_or_Expectation,Manipulated_Factor,...,Failure_or_Warning_Pattern,Threats_to_Validity,Reporting_Destination,Interpretation_Boundary,Status,Owner,Outcome_Summary,Decision_Taken,Notes,Experiment_Ready
1,E02,Task pooling experiment,Target-definition,Secondary decision-support,Stage 1 - Target lock,Medium,Task pooling policy,Should the main target be learned across all t...,Evaluate under leakage-aware claim-matched spl...,Pooling strategy by task,...,"Weak-split inflation, instability, class colla...","Class imbalance, sample limits, domain shift, ...",Chapter 3 Method choice,Interpret only under stated split logic/eviden...,Planned,Khaled (thesis author),NaN,NaN,NaN,NaN



Decision_Log rows mentioning E02:


,Decision_ID,Decision_Topic,Stage,Candidate_Options,Evidence_Experiments,Decision_Criteria,Chosen_Option,Why_Chosen,Why_Not_Others,Risks_or_Tradeoffs,Grounding_Section_or_Source_Packet,Date_Locked,Freeze_Status,Thesis_Use
0,D01,Final target choice,Stage 1 - Target lock,fine emotion; coarse_affect; binary valence-like,"E01,E02,E03",Construct validity + class balance + per-subje...,To be locked after method-choice evidence review,NaN,NaN,"Tradeoff between construct validity, label gra...",To be linked after background/method revision,NaN,Open,Chapter 3 method lock feeding Chapter 4 confir...



Grouping_Strategy rows mentioning E02:


,Grouping_Strategy_ID,Strategy_Name,Split_Family,Train_Group_Unit,Test_Group_Unit,Grouping_Level,Train_Rule,Test_Rule,Leakage_Safeguard,Suitable_For,Not_Suitable_For,Interpretation_Boundary,Typical_Experiments,Thesis_Section,Notes
1,GS02,pooled-task within-subject LOSO-session,within_subject,subject_session,session,within_subject,"Within each subject, pool tasks and hold out o...",Evaluate on held-out session with pooled task ...,No test-session information in pooled train tr...,confirmatory_within_person,confirmatory_cross_person_transfer,Within-person conclusion under pooled task pol...,"E02,E16",Chapter 3 Method choice,NaN
2,GS03,task-specific within-subject LOSO-session,within_subject,subject_session,session,task_level,"Within each subject/task, hold out one session.",Test on same-task held-out session.,Task-specific split manifests and train-only p...,method_choice,confirmatory_cross_person_transfer,Task-conditioned within-person evidence only.,"E02,E15",Chapter 3 Method choice,NaN



Data_Selection rows relevant to pooled vs task-specific framing:


,Data_Slice_ID,Slice_Name,Purpose,Subject_Scope,Session_Scope,Time_Window,Task_Scope,Modality_Scope,Target_Type,Label_Set,Inclusion_Rule,Exclusion_Rule,Class_Balance_Policy,Minimum_Samples_Rule,Leakage_Risk,Valid_Use_Case,Thesis_Use,Notes
0,DS01,"All subjects, all sessions, pooled task, poole...",Global baseline for target/split/model method-...,all_subjects,all_sessions,full_study_window,pooled_all_tasks,pooled_all_modalities,coarse_affect,coarse_affect,All QC-passed repeated-session beta maps with ...,Drop failed-preprocessing rows and missing tar...,as_is,Prefer >=20 per class before claim-level inter...,medium,method_choice,method_choice,NaN
2,DS03,Task-specific pooled-modality,Task pooling vs task-specific method-choice an...,all_subjects,all_sessions,full_study_window,task_specific_other,pooled_all_modalities,coarse_affect,coarse_affect by task,Restrict each run to one task family.,Exclude off-task records for that run.,as_is,Minimum counts checked per task-specific slice.,medium,method_choice,method_choice,NaN


### Workbook correctness check

Use this logic:

- If E02 in `Experiment_Definitions` is still disabled and/or uses `filter_task = VARIES` with no executable expansion attached, then the **workbook path is not ready**.
- If `Master_Experiments` and `Grouping_Strategy_Map` clearly distinguish:
  - pooled-task within-subject LOSO
  - task-specific within-subject LOSO  
  then the **scientific framing is present**, even if the workbook execution row is incomplete.


## 3. Dataset audit – what tasks are actually in the index?

This is the most important readiness check for E02.

The registry currently expands task-specific E02 over whatever task labels are present in the dataset index. That is convenient operationally, but it is only scientifically correct if those task labels are exactly the intended task families and they have enough support.


In [4]:
df = pd.read_csv(INDEX_CSV)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Shape: (3402, 17)
Columns: ['sample_id', 'subject', 'session', 'bas', 'run', 'task', 'emotion', 'coarse_affect', 'modality', 'beta_index', 'beta_file', 'beta_path', 'mask_path', 'regressor_label', 'raw_label', 'subject_session', 'subject_session_bas']


,sample_id,subject,session,bas,run,task,emotion,coarse_affect,modality,beta_index,beta_file,beta_path,mask_path,regressor_label,raw_label,subject_session,subject_session_bas
0,sub-001_ses-01_BAS2_run-1_beta-1,sub-001,ses-01,BAS2,1,passive,anger,negative,audio,1,beta_0001.nii,sub-001/ses-01/BAS2/beta_0001.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_audio,run-1_passive_anger_audio,sub-001_ses-01,sub-001_ses-01_BAS2
1,sub-001_ses-01_BAS2_run-1_beta-2,sub-001,ses-01,BAS2,1,passive,anger,negative,video,2,beta_0002.nii,sub-001/ses-01/BAS2/beta_0002.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_video,run-1_passive_anger_video,sub-001_ses-01,sub-001_ses-01_BAS2
2,sub-001_ses-01_BAS2_run-1_beta-3,sub-001,ses-01,BAS2,1,passive,anger,negative,audiovisual,3,beta_0003.nii,sub-001/ses-01/BAS2/beta_0003.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anger_audiovisual,run-1_passive_anger_audiovisual,sub-001_ses-01,sub-001_ses-01_BAS2
3,sub-001_ses-01_BAS2_run-1_beta-4,sub-001,ses-01,BAS2,1,passive,anxiety,negative,audio,4,beta_0004.nii,sub-001/ses-01/BAS2/beta_0004.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anxiety_audio,run-1_passive_anxiety_audio,sub-001_ses-01,sub-001_ses-01_BAS2
4,sub-001_ses-01_BAS2_run-1_beta-5,sub-001,ses-01,BAS2,1,passive,anxiety,negative,video,5,beta_0005.nii,sub-001/ses-01/BAS2/beta_0005.nii,sub-001/ses-01/BAS2/mask.nii,run-1_passive_anxiety_video,run-1_passive_anxiety_video,sub-001_ses-01,sub-001_ses-01_BAS2


In [5]:
required_cols = ["subject", "session", "task", "coarse_affect"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns for E02 audit: {missing}")

print("Subjects:", sorted(df["subject"].dropna().astype(str).unique().tolist()))
print("Tasks:", sorted(df["task"].dropna().astype(str).unique().tolist()))
print("Sessions:", sorted(df["session"].dropna().astype(str).unique().tolist())[:20])


Subjects: ['sub-001', 'sub-002']
Tasks: ['emo', 'passive', 'recog']
Sessions: ['ses-01', 'ses-02', 'ses-03', 'ses-04', 'ses-05', 'ses-06', 'ses-07', 'ses-08', 'ses-09', 'ses-10', 'ses-11', 'ses-12', 'ses-13', 'ses-14', 'ses-15', 'ses-16', 'ses-17', 'ses-18', 'ses-19', 'ses-20']


## 4. Pooled-task baseline audit

This shows the basic data support for the pooled-task side of E02.


In [6]:
pooled = df.dropna(subset=["coarse_affect"]).copy()

print("Overall pooled-task class counts:")
display(pooled["coarse_affect"].value_counts(dropna=False).rename_axis("class").to_frame("count"))

print("Class counts by subject:")
display(
    pooled.groupby(["subject", "coarse_affect"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)


Overall pooled-task class counts:


,count
class,
negative,1512
positive,1512
neutral,378


Class counts by subject:


coarse_affect,negative,neutral,positive
subject,,,
sub-001,684,171,684
sub-002,828,207,828


## 5. Task-specific audit

This is the critical correctness check.

For E02 to be fair, each task-specific slice should be evaluated only if it has enough samples and enough class coverage.


In [7]:
task_subject_counts = (
    pooled.groupby(["subject", "task", "coarse_affect"])
    .size()
    .rename("count")
    .reset_index()
)

display(task_subject_counts.sort_values(["subject", "task", "coarse_affect"]))


,subject,task,coarse_affect,count
0,sub-001,emo,negative,228
1,sub-001,emo,neutral,57
2,sub-001,emo,positive,228
3,sub-001,passive,negative,228
4,sub-001,passive,neutral,57
5,sub-001,passive,positive,228
6,sub-001,recog,negative,228
7,sub-001,recog,neutral,57
8,sub-001,recog,positive,228
9,sub-002,emo,negative,276


In [8]:
task_summary = (
    pooled.groupby(["subject", "task"])
    .agg(
        n_rows=("coarse_affect", "size"),
        n_classes=("coarse_affect", "nunique"),
        n_sessions=("session", "nunique"),
    )
    .reset_index()
    .sort_values(["subject", "task"])
)

display(task_summary)


,subject,task,n_rows,n_classes,n_sessions
0,sub-001,emo,513,3,19
1,sub-001,passive,513,3,19
2,sub-001,recog,513,3,19
3,sub-002,emo,621,3,23
4,sub-002,passive,621,3,23
5,sub-002,recog,621,3,23


### Minimum-support threshold

The workbook's `Data_Profile` sheet uses a sparse-count threshold of **20** as the default diagnostic threshold.

Use that as an initial planning threshold here, but treat it as a governance threshold, not an absolute law.


In [9]:
MIN_COUNT_PER_CLASS = 20

class_counts = (
    pooled.groupby(["subject", "task", "coarse_affect"])
    .size()
    .rename("count")
    .reset_index()
)

task_support = (
    class_counts.groupby(["subject", "task"])
    .agg(
        min_class_count=("count", "min"),
        max_class_count=("count", "max"),
        n_classes=("count", "size"),
        total_rows=("count", "sum"),
    )
    .reset_index()
    .sort_values(["subject", "task"])
)

task_support["meets_min_count_per_class"] = task_support["min_class_count"] >= MIN_COUNT_PER_CLASS
display(task_support)


,subject,task,min_class_count,max_class_count,n_classes,total_rows,meets_min_count_per_class
0,sub-001,emo,57,228,3,513,True
1,sub-001,passive,57,228,3,513,True
2,sub-001,recog,57,228,3,513,True
3,sub-002,emo,69,276,3,621,True
4,sub-002,passive,69,276,3,621,True
5,sub-002,recog,69,276,3,621,True


## 6. LOSO-session viability for task-specific E02

This checks, for each `(subject, task)` slice, whether held-out-session evaluation is even structurally viable.


In [10]:
def loso_viability_by_subject_task(df: pd.DataFrame, target: str = "coarse_affect") -> pd.DataFrame:
    rows = []
    valid = df.dropna(subset=[target]).copy()
    for (subject, task), sdf in valid.groupby(["subject", "task"]):
        all_classes = set(sdf[target].dropna().unique().tolist())
        sessions = sorted(sdf["session"].dropna().unique().tolist())
        for heldout_session in sessions:
            train = sdf[sdf["session"] != heldout_session]
            test = sdf[sdf["session"] == heldout_session]
            train_classes = set(train[target].dropna().unique().tolist())
            test_classes = set(test[target].dropna().unique().tolist())
            rows.append({
                "subject": subject,
                "task": task,
                "heldout_session": heldout_session,
                "n_train": len(train),
                "n_test": len(test),
                "n_all_classes": len(all_classes),
                "n_train_classes": len(train_classes),
                "n_test_classes": len(test_classes),
                "train_has_all_global_classes": train_classes == all_classes,
                "missing_from_train": sorted(test_classes - train_classes),
            })
    return pd.DataFrame(rows)

viability = loso_viability_by_subject_task(pooled, target="coarse_affect")
display(viability.head(20))


,subject,task,heldout_session,n_train,n_test,n_all_classes,n_train_classes,n_test_classes,train_has_all_global_classes,missing_from_train
0,sub-001,emo,ses-01,486,27,3,3,3,True,[]
1,sub-001,emo,ses-02,486,27,3,3,3,True,[]
2,sub-001,emo,ses-03,486,27,3,3,3,True,[]
3,sub-001,emo,ses-04,486,27,3,3,3,True,[]
4,sub-001,emo,ses-05,486,27,3,3,3,True,[]
5,sub-001,emo,ses-06,486,27,3,3,3,True,[]
6,sub-001,emo,ses-07,486,27,3,3,3,True,[]
7,sub-001,emo,ses-08,486,27,3,3,3,True,[]
8,sub-001,emo,ses-09,486,27,3,3,3,True,[]
9,sub-001,emo,ses-10,486,27,3,3,3,True,[]


In [11]:
problem_folds = viability[
    (~viability["train_has_all_global_classes"]) |
    (viability["n_train"] == 0) |
    (viability["n_test"] == 0)
].copy()

print("Problematic task-specific LOSO folds:")
display(problem_folds.sort_values(["subject", "task", "heldout_session"]))


Problematic task-specific LOSO folds:


,subject,task,heldout_session,n_train,n_test,n_all_classes,n_train_classes,n_test_classes,train_has_all_global_classes,missing_from_train


## 7. E02 readiness conclusion helper

This cell produces a simple operational/scientific diagnosis.


In [12]:
def readiness_summary():
    lines = []

    # Registry readiness
    if e02_entry and e02_entry.get("executable_now") and e02_entry.get("execution_status") == "executable":
        lines.append("Registry path: READY")
    else:
        lines.append("Registry path: NOT READY")

    # Workbook readiness
    e02_row = exp_defs[exp_defs["experiment_id"].astype(str).str.upper() == "E02"]
    workbook_ready = False
    if len(e02_row):
        row = e02_row.iloc[0].to_dict()
        enabled = str(row.get("enabled", "")).strip().lower() == "yes"
        # Treat VARIES in filter_task as non-executable until expanded.
        filter_task = str(row.get("filter_task", "")).strip()
        workbook_ready = enabled and filter_task not in {"", "VARIES", "varies", "nan"}
    lines.append(f"Workbook path: {'READY' if workbook_ready else 'NOT READY'}")

    # Scientific sufficiency
    if len(task_support):
        all_ok = task_support["meets_min_count_per_class"].all()
        any_problem_folds = len(problem_folds) > 0
        if all_ok and not any_problem_folds:
            lines.append("Task-specific scientific support: STRONG")
        elif any(task_support["meets_min_count_per_class"]):
            lines.append("Task-specific scientific support: PARTIAL")
        else:
            lines.append("Task-specific scientific support: WEAK")
    else:
        lines.append("Task-specific scientific support: UNKNOWN")

    return lines

for line in readiness_summary():
    print("-", line)


- Registry path: READY
- Workbook path: NOT READY
- Task-specific scientific support: STRONG


## 8. Dry-run command for E02

This should be your first real operator step after the audit.


In [13]:
dry_run_cmd = [
    sys.executable, "-m", "uv", "run", "--active",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
    "--dry-run",
]
print(" ".join(f'"{part}"' if " " in part else part for part in dry_run_cmd))


d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe -m uv run --active thesisml-run-decision-support --registry D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json --experiment-id E02 --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --output-root D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support --dry-run


In [14]:
# Uncomment to execute the dry-run.
result = subprocess.run(dry_run_cmd, text=True, capture_output=True)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)


Return code: 0
Campaign complete
- campaign_id: 20260324_104533_443247
- campaign_root: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104533_443247
- selected_experiments: E02
- status_counts: {'dry_run': 8}
- run_log_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104533_443247\run_log_export.csv
- decision_summary: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104533_443247\decision_support_summary.csv
- decision_report: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104533_443247\decision_recommendations.md
- aggregation: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104533_443247\result_aggregation.json
- summary_outputs_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104533_443247\summary_outputs_export.csv
- manifest: D:\Khaled

## 9. Smoke-run recommendation

Before running the full E02 batch, do two smoke runs:

1. pooled-task baseline
2. one task-specific variant with the best support

This notebook picks the best-supported task automatically from the audit.


In [15]:
# Choose a strong candidate task for smoke testing.
good_tasks = task_support.sort_values(
    ["meets_min_count_per_class", "min_class_count", "total_rows"],
    ascending=[False, False, False]
)
display(good_tasks)

if len(good_tasks):
    best_subject = good_tasks.iloc[0]["subject"]
    best_task = good_tasks.iloc[0]["task"]
    print("Suggested smoke subject:", best_subject)
    print("Suggested smoke task:", best_task)
else:
    best_subject = None
    best_task = None


,subject,task,min_class_count,max_class_count,n_classes,total_rows,meets_min_count_per_class
3,sub-002,emo,69,276,3,621,True
4,sub-002,passive,69,276,3,621,True
5,sub-002,recog,69,276,3,621,True
0,sub-001,emo,57,228,3,513,True
1,sub-001,passive,57,228,3,513,True
2,sub-001,recog,57,228,3,513,True


Suggested smoke subject: sub-002
Suggested smoke task: emo


In [16]:
if best_subject is not None:
    pooled_smoke_cmd = [
        "thesisml-run-experiment",
        "--index-csv", str(INDEX_CSV),
        "--data-root", str(DATA_ROOT),
        "--cache-dir", str(CACHE_DIR),
        "--target", "coarse_affect",
        "--model", "ridge",
        "--cv", "within_subject_loso_session",
        "--subject", str(best_subject),
        "--run-id", f"e02_pooled_smoke_{best_subject}",
    ]
    print("Pooled smoke command:")
    print(" ".join(f'"{part}"' if " " in part else part for part in pooled_smoke_cmd))

    task_smoke_cmd = [
        "thesisml-run-experiment",
        "--index-csv", str(INDEX_CSV),
        "--data-root", str(DATA_ROOT),
        "--cache-dir", str(CACHE_DIR),
        "--target", "coarse_affect",
        "--model", "ridge",
        "--cv", "within_subject_loso_session",
        "--subject", str(best_subject),
        "--filter-task", str(best_task),
        "--run-id", f"e02_task_specific_smoke_{best_subject}_{best_task}",
    ]
    print("\nTask-specific smoke command:")
    print(" ".join(f'"{part}"' if " " in part else part for part in task_smoke_cmd))


Pooled smoke command:
thesisml-run-experiment --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --target coarse_affect --model ridge --cv within_subject_loso_session --subject sub-002 --run-id e02_pooled_smoke_sub-002

Task-specific smoke command:
thesisml-run-experiment --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --target coarse_affect --model ridge --cv within_subject_loso_session --subject sub-002 --filter-task emo --run-id e02_task_specific_smoke_sub-002_emo


## 10. Full E02 launch command

Run this only after:

- dry-run succeeds
- pooled baseline looks healthy
- task-specific slices are scientifically supportable


In [17]:
full_cmd = [
    sys.executable, "-m", "uv", "run",
    "--active",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
]
print(" ".join(f'"{part}"' if " " in part else part for part in full_cmd))


d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe -m uv run --active thesisml-run-decision-support --registry D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json --experiment-id E02 --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --output-root D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support


In [18]:
# Uncomment to execute the full E02 batch.
run_cmd = [
    sys.executable, "-m", "uv", "run", "--active",
    "thesisml-run-decision-support",
    "--registry", str(REGISTRY),
    "--experiment-id", EXPERIMENT_ID,
    "--index-csv", str(INDEX_CSV),
    "--data-root", str(DATA_ROOT),
    "--cache-dir", str(CACHE_DIR),
    "--output-root", str(OUTPUT_ROOT),
]
print(" ".join(f'\"{part}\"' if " " in part else part for part in run_cmd))
result = subprocess.run(run_cmd, text=True, capture_output=True)
print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)


d:\Khaled\Projects\VScode\Thesis\.venv-gpu\Scripts\python.exe -m uv run --active thesisml-run-decision-support --registry D:\Khaled\Projects\VScode\Thesis\configs\decision_support_registry.json --experiment-id E02 --index-csv D:\Khaled\Projects\VScode\Thesis\Data\processed\dataset_index.csv --data-root D:\Khaled\Projects\VScode\Thesis\Data --cache-dir D:\Khaled\Projects\VScode\Thesis\Data\processed\feature_cache --output-root D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support
Return code: 0
Campaign complete
- campaign_id: 20260324_104540_038453
- campaign_root: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104540_038453
- selected_experiments: E02
- status_counts: {'completed': 8}
- run_log_export: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104540_038453\run_log_export.csv
- decision_summary: D:\Khaled\Projects\VScode\Thesis\outputs\artifacts\decision_support\campaigns\20260324_104540

## 11. Correctness interpretation guide for E02

E02 is correct only if you interpret it this way:

- pooled-task = one within-subject LOSO-session model using all tasks together
- task-specific = separate within-subject LOSO-session runs per task and subject
- a task-specific win is meaningful only if:
  - the task slices are adequately supported
  - class collapse does not drive the result
  - the compared tasks are the intended task families

### What would make E02 misleading?

- including tasks that were not intended in the scientific design
- comparing pooled-task against task slices that are too sparse
- interpreting higher task-specific scores as better methodology when they mainly reflect easier or smaller subsets


## 12. Practical conclusion template

Use the audit results to classify E02 as one of these:

### Case A — ready and correct
- registry path ready
- workbook path either fixed or intentionally not used
- task-specific slices are sufficiently supported
- LOSO viability is acceptable

### Case B — operationally ready but scientifically partial
- registry path ready
- task-specific slices exist
- some tasks are too sparse or unstable
- E02 should be narrowed to a task whitelist or interpreted cautiously

### Case C — not ready
- workbook/registry mismatch plus sparse tasks plus LOSO failure
- E02 should not be run fully until the task scope is repaired


## 13. Recommended next step

If E02 is only partially correct because some tasks are too weak, the cleanest fix is usually **not** core-engine code.
It is usually one of:

- restrict E02 to a pre-approved task whitelist in the registry
- or make workbook execution explicit with a controlled `Search_Spaces` / custom-matrix design
- or document that E02 is exploratory and descriptive rather than a strict apples-to-apples comparison
